<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# **Operaciones de Aprendizaje Automático III**

# **Clase 2, Anatomía de un pipeline de inferencia**

Cuando escribimos `respuesta = modelo.generate(prompt)` esa línea esconde ocho etapas:

1. PROMPT TEMPLATE: nuestra lógica de negocio (la escribimos nosotros)
2. CHAT TEMPLATE: viene en el repo del modelo (no es nuestro)
3. TOKENIZACIÓN: texto a ids enteros
4. PREFILL: una pasada ids a distribución sobre el vocabulario
5. SAMPLING: de la distribución elegimos un token (temperatura, top-k, top-p, penalizaciones)
6. DECODE (bucle): repetir 4-5 hasta EOS o max_tokens
7. DETOKENIZACIÓN: ids a texto
8. VALIDACIÓN: ¿el texto sirve para lo que sigue?

## **a) Instalación y entorno**

Antes de importar transformers / huggingface_hub

`huggingface_hub` resuelve las rutas de caché **en tiempo de import* (`huggingface_hub.constants`). Si seteamos `HF_HOME` después de `import transformers`, la variable no tiene ningún efecto: los pesos igual van a caer en `~/.cache/huggingface`.

In [ ]:
import os
os.environ["HF_HOME"] = "/content/hf_cache"

In [ ]:
import json, textwrap, hashlib, time, re, platform

import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import constants as hf_constants

Verificamos, si esto no apunta a /content/hf_cache, la celda 1 se ejecutó tarde

In [ ]:
print(f"HF_HOME efectivo : {hf_constants.HF_HOME}")
print(f"caché del hub: {hf_constants.HF_HUB_CACHE}")
assert hf_constants.HF_HOME.startswith("/content"), (
    "HF_HOME no tomó efecto: reiniciar la sesión y correr la celda 1 primero."
)

HF_HOME efectivo : /content/hf_cache
caché del hub    : /content/hf_cache/hub


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"device: {DEVICE}")
print(f"dtype: {DTYPE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    cc = torch.cuda.get_device_capability(0)
    print(f"compute cap. : {cc[0]}.{cc[1]}")

torch: 2.11.0+cu128
transformers: 5.16.1
device: cuda
dtype: torch.float16
GPU          : Tesla T4
compute cap. : 7.5


**Detalle importante:** `cu128` (CUDA 12.8) y *compute capability* 7.5 no son lo mismo.
El primero es la versión de CUDA contra la que está compilado PyTorch, el segundo
describe la arquitectura de la GPU. Son dos campos distintos del artefacto.

- Usar la misma *seed* en CPU con `float32` y en GPU con `float16` **no** garantiza el
  mismo texto: cambia la precisión y cambia el orden de las reducciones.
- Por eso la seed sola no alcanza: hay que registrar también `device` y `dtype`.

## **b) Cargar el modelo, y anotar que modelo exactamente**

El nombre del repositorio identifica el repo, no una versión inmutable de sus archivos, main es una rama y se mueve, el commit SHA identifica una revisión concreta.

El tokenizer y el modelo se descargan por separado. Si pinneamos solo una, podemos terminar con pesos de un commit y un chat template de otro. El modelo va a seguir respondiendo peor, y sin ningún error.

In [ ]:
REPO = "Qwen/Qwen2.5-1.5B-Instruct"

Averiguamos el SHA de main hoy, para pinnearlo. Corremos esta celda una vez, copiamos el SHA que imprime y lo pegamos en revisión (celda siguiente) a partir de ahí queda congelado.

In [ ]:
from huggingface_hub import HfApi

sha_main = HfApi().model_info(REPO).sha
print(f"SHA de main AHORA: {sha_main}")
print(f"IMPORTANTE, Copiamos esto {sha_main}")

SHA de main AHORA: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306
IMPORTANTE, Copiamos esto 989aa7980e4cf806f80c7fef2b1adb7bc71aa306


Pegamos el SHA en este bloque.

El SHA corto funciona, pero se registra el completo (40 caracteres): un prefijo corto puede volverse ambiguo cuando el repositorio crece.

In [ ]:
REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"  # SHA completo

In [ ]:
tok = AutoTokenizer.from_pretrained(REPO, revision=REVISION)

try:
    modelo = AutoModelForCausalLM.from_pretrained(REPO, revision=REVISION, dtype=DTYPE)
except TypeError:
    modelo = AutoModelForCausalLM.from_pretrained(REPO, revision=REVISION, torch_dtype=DTYPE)

modelo = modelo.to(DEVICE)
modelo.eval()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

* El SHA que se registra es el que quedó en disco
* Si main se movió entre la descarga y esa consulta, estaríamos registrando un SHA que no corresponde a lo que corrimos

* La caché de Hugging Face guarda cada revisión. El nombre de la carpeta es el SHA real. Eso es lo que se registra.

In [ ]:
from huggingface_hub import snapshot_download

Ya está descargado, esto solo resuelve la ruta local no vuelve a bajar nada


Lo siguiente garantiza la trazabilidad del modelo: obtiene la revisión exacta que está en la caché de Hugging Face, extrae su SHA y registra además características como la arquitectura, número de capas, tipo de datos y cantidad de parámetros

In [ ]:
ruta_snapshot = snapshot_download(REPO, revision=REVISION, allow_patterns=["config.json"])
SHA_EN_DISCO  = os.path.basename(ruta_snapshot)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
print(f"Snapshot: {ruta_snapshot}")
print(f"SHA en disco: {SHA_EN_DISCO}  (esto va al registry)")

Snapshot: /content/hf_cache/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306
SHA en disco: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306  (esto va al registry)


In [ ]:
cfg = modelo.config

print(f"Repositorio: {REPO}")
print(f"Revisión (SHA): {SHA_EN_DISCO}")
print(f"Arquitectura: {cfg.architectures[0] if cfg.architectures else '?'}")
print(f"Capas: {cfg.num_hidden_layers}")
print(f"dtype cargado: {next(modelo.parameters()).dtype}")
print(f"Parámetros: {sum(p.numel() for p in modelo.parameters())/1e9:.2f} B")

Repositorio: Qwen/Qwen2.5-1.5B-Instruct
Revisión (SHA): 989aa7980e4cf806f80c7fef2b1adb7bc71aa306
Arquitectura: Qwen2ForCausalLM
Capas: 28
dtype cargado: torch.float16
Parámetros: 1.54 B


El modelo usado es Qwen2.5-1.5B-Instruct, revisión SHA específica, con 28 capas, 151.665 tokens de vocabulario, 1.54B parámetros y cargado en FP16.

El SHA es lo que registramos para identificar la versión exacta del modelo, si mañana main apunta a un nuevo commit, el pipeline podría usar una versión diferente sin cambiar el código.

---

`len(tok)` no es `config.vocab_size` y la diferencia se ve en los logits:

- `tok.vocab_size`: el vocabulario base del tokenizer
- `len(tok)`: vocabulario base + tokens añadidos (los especiales de chat)
- `config.vocab_size`: filas de la matriz de salida del modelo

El último suele ser mayor: la matriz de embeddings se rellena hasta un múltiplo
(típicamente 128 o 256) porque así los kernels de GPU son más eficientes.

Consecuencia operativa: hay filas de logits que no corresponden a ningún token.


Si sampleamos sin filtrar, podemos caer en un id que al decodificar no devuelve nada.

In [ ]:
print(f"tok.vocab_size: {tok.vocab_size:,} (vocabulario base)")
print(f"len(tok): {len(tok):,} (+ tokens añadidos)")
print(f"modelo.config.vocab_size: {cfg.vocab_size:,} (filas de la capa de salida)")
print(f"\nfilas de logits sin token asociado: {cfg.vocab_size - len(tok):,}")

tok.vocab_size: 151,643 (vocabulario base)
len(tok): 151,665 (+ tokens añadidos)
modelo.config.vocab_size: 151,936 (filas de la capa de salida)

filas de logits sin token asociado: 271


---

`generation_config.json`: el sampling que no elegimos

Este archivo viene en el repo del modelo, escrito por quien lo publicó. Si no pasamos parámetros explícitos, el comportamiento de nuestra aplicación lo está decidiendo un archivo de un tercero que además puede cambiar con un commit.

Acá también aparece algo que rompe bucles: eos_token_id puede ser una lista. Qwen2.5-Instruct declara dos finales posibles. Un if token == tok.eos_token_id solo detecta uno de ellos, si sale el otro, la generación no para y sigue escribiendo después del final.


In [ ]:
print("Defaults de sampling que trae el proveedor:")
print(json.dumps(modelo.generation_config.to_diff_dict(), indent=2, ensure_ascii=False))

# Conjunto de EOS, siempre construirlo así, no comparar contra un solo id.
_eos = modelo.generation_config.eos_token_id
EOS_IDS = set(_eos) if isinstance(_eos, (list, tuple)) else {_eos}
EOS_IDS.discard(None)

print("EOS ids:", {i: tok.convert_ids_to_tokens(i) for i in EOS_IDS})

Defaults de sampling que trae el proveedor:
{
  "do_sample": true,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8,
  "repetition_penalty": 1.1,
  "pad_token_id": 151643,
  "bos_token_id": 151643,
  "eos_token_id": [
    151645,
    151643
  ],
  "transformers_version": "5.16.1"
}
EOS ids: {151643: '<|endoftext|>', 151645: '<|im_end|>'}


##**c) Etapa 1: el prompt template**

Un prompt template es una plantilla de texto que define cómo le vamos a pedir algo al modelo, dejando espacios variables para insertar los datos reales.

El prompt template es una capa propia del proyecto que convierte los datos del negocio en texto, se versiona con Git y cada cambio en el prompt debe tratarse como un cambio real del artefacto, al igual que cambiar de modelo.

In [ ]:
PLANTILLA_SISTEMA = (
    "Eres un asistente de atención al cliente de una tienda online argentina. "
    "Respondes en español, en dos oraciones como máximo, "
    "sin inventar políticas que no te hayan dado."
)

PLANTILLA_USUARIO = (
    "Reclamo de un cliente:\n"
    "- Producto: {producto}\n"
    "- Días desde la compra: {dias}\n"
    "- Problema: {problema}\n\n"
    "Indica si corresponde gestionar el reclamo y qué debe hacer el cliente."
)

CASO = {
    "producto": "auriculares inalámbricos",
    "dias": 12,
    "problema": "el auricular derecho dejó de cargar",
}

texto_usuario = PLANTILLA_USUARIO.format(**CASO)

**Etapa 1, Prompt Template (capa nuestra)**

In [ ]:
print(texto_usuario)

Reclamo de un cliente:
- Producto: auriculares inalámbricos
- Días desde la compra: 12
- Problema: el auricular derecho dejó de cargar

Indica si corresponde gestionar el reclamo y qué debe hacer el cliente.


El hash del template es su huella digital: si cambia una coma, cambia el hash. Sirve para identificar la versión exacta del prompt con la que se produjo un resultado.


Hasheamos las plantillas, no el texto renderizado: lo que se versiona es la plantilla; los datos del caso cambian en cada request.

In [ ]:
hash_prompt = hashlib.sha256(
    (PLANTILLA_SISTEMA + "||" + PLANTILLA_USUARIO).encode()
).hexdigest()[:12]
print(f"Hash del prompt template: {hash_prompt}")

Hash del prompt template: f711814fec4a


##**d) Etapa 2, el Chat Template (no es nuestro)**

Prompt template no es lo mismo que chat template:

| | Prompt template | Chat template |
|---|---|---|
| Quién lo escribe | Nosotros | Quien publicó el modelo |
| Dónde vive | Nuestro git | `tokenizer_config.json` del repo |
| Formato | Texto con `{variables}` | Template Jinja |
| Cuándo cambia | Cuando lo cambiamos | Con un commit ajeno |

El chat template determina cómo se representan los turnos (`system`, `user`, `assistant`)
y dónde van los tokens especiales de control. Si lo aplicamos mal, o no lo aplicamos
cuando el modelo lo necesita, la calidad cae sin que se produzca ningún error.

In [ ]:
mensajes = [
    {"role": "system", "content": PLANTILLA_SISTEMA},
    {"role": "user",   "content": texto_usuario},
]

tokenize=False nos devuelve el string exacto que se va a tokenizar, es la forma de ver qué está pasando de verdad.

In [ ]:
texto_formateado = tok.apply_chat_template(
    mensajes, tokenize=False, add_generation_prompt=True
)

**Etapa 2, Chat Template (capa del modelo, no nuestra)**

String exacto que recibirá el tokenizador (con los tokens especiales):

In [ ]:
print(repr(texto_formateado)[:1400])

'<|im_start|>system\nEres un asistente de atención al cliente de una tienda online argentina. Respondes en español, en dos oraciones como máximo, sin inventar políticas que no te hayan dado.<|im_end|>\n<|im_start|>user\nReclamo de un cliente:\n- Producto: auriculares inalámbricos\n- Días desde la compra: 12\n- Problema: el auricular derecho dejó de cargar\n\nIndica si corresponde gestionar el reclamo y qué debe hacer el cliente.<|im_end|>\n<|im_start|>assistant\n'


`add_generation_prompt=True`: la conversación ya terminó y ahora es tu turno, empieza a generar la respuesta del asistente

Sin eso, el modelo no sabe que le toca hablar a él. Comparemos los úlltimos caracteres:


In [ ]:
sin_gen = tok.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=False)

print("con add_generation_prompt:", repr(texto_formateado[-60:]))
print("sin add_generation_prompt:", repr(sin_gen[-60:]))

con add_generation_prompt: ' qué debe hacer el cliente.<|im_end|>\n<|im_start|>assistant\n'
sin add_generation_prompt: 'gestionar el reclamo y qué debe hacer el cliente.<|im_end|>\n'


Descargamos solo el tokenizador de otro modelo (unos pocos MB) para ver que el formato no es universal.

In [ ]:
REPO_OTRO = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
REVISION_OTRO = "fe8a4ea1ffedaf415f4da2f062534de366a451e6"   # mismo criterio: pinnear con el SHA

tok_otro = None
try:
    tok_otro = AutoTokenizer.from_pretrained(REPO_OTRO, revision=REVISION_OTRO)
    otro = tok_otro.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)

    print("Qwen2.5   :", repr(texto_formateado[:110]), "...")
    print("TinyLlama :", repr(otro[:110]), "...")
    print("\nMismo contenido, marcado completamente distinto.")
except Exception as e:
    print(f"(no se pudo descargar el segundo tokenizador: {e})")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Qwen2.5   : '<|im_start|>system\nEres un asistente de atención al cliente de una tienda online argentina. Respondes en españ' ...
TinyLlama : '<|system|>\nEres un asistente de atención al cliente de una tienda online argentina. Respondes en español, en d' ...

Mismo contenido, marcado completamente distinto.


Migrar de modelo no es cambiar un string en el config: es cambiar el formato del input. Por eso el chat template es campo del artefacto.

##**e) Etapa 3, Tokenización**


El modelo no ve texto, ve enteros, la tabla que traduce texto a enteros, es el tokenizador, y es específica de ese modelo con ese vocabulario.

Pasamos `attention_mask` explícitamente. Con batch de 1 y sin padding no cambia el resultado, pero evita un warning de transformers y es la práctica correcta: la máscara le dice al modelo qué posiciones son reales y cuáles son relleno.

In [ ]:
enc = tok(texto_formateado, return_tensors="pt", add_special_tokens=False).to(DEVICE)
input_ids = enc["input_ids"]
attention_mask = enc["attention_mask"]

n_chars = len(texto_formateado)
n_tokens = input_ids.shape[1]

print(f"Caracteres del string: {n_chars}")
print(f"Tokens resultantes: {n_tokens}")
print(f"Ratio caracteres/token: {n_chars / n_tokens:.2f}")

Caracteres del string: 456
Tokens resultantes: 115
Ratio caracteres/token: 3.97


En español el ratio es peor que en inglés: el mismo contenido cuesta más tokens. Eso impacta en tres lugares a la vez: costo (se paga por token), latencia (el prefill es cuadrático en la longitud) y ventana de contexto (entra menos).

Es una de las razones por las que un sistema en español no se dimensiona con los benchmarks publicados en inglés.

In [ ]:
print(f"  {'#':>3}  {'id':>7}  {'token (repr)':<26}")
print("  " + "-" * 42)
for i, t in enumerate(input_ids[0][:18].tolist()):
    print(f"  {i:>3}  {t:>7}  {repr(tok.convert_ids_to_tokens(t)):<26}")

    #       id  token (repr)              
  ------------------------------------------
    0   151644  '<|im_start|>'            
    1     8948  'system'                  
    2      198  'Ċ'                       
    3       36  'E'                       
    4      416  'res'                     
    5      650  'Ġun'                     
    6      438  'Ġas'                     
    7      380  'ist'                     
    8     6817  'ente'                    
    9      409  'Ġde'                     
   10    80541  'ĠatenciÃ³n'              
   11      452  'Ġal'                     
   12    26629  'Ġcliente'                
   13      409  'Ġde'                     
   14     5093  'Ġuna'                    
   15     8988  'Ġti'                     
   16     9696  'enda'                    
   17     2860  'Ġonline'                 


In [ ]:
print("Tokens especiales de este modelo:")
for nombre in ("bos_token", "eos_token", "pad_token", "unk_token"):
    val = getattr(tok, nombre, None)
    tid = getattr(tok, nombre + "_id", None)
    print(f"{nombre:<12}: {str(val):<20} id={tid}")

Tokens especiales de este modelo:
bos_token   : None                 id=None
eos_token   : <|im_end|>           id=151645
pad_token   : <|endoftext|>        id=151643
unk_token   : None                 id=None


El tokenizer usa <|im_end|> (ID 151645) como token de fin, <|endoftext|> (ID 151643) como padding y no tiene BOS ni UNK explícitos configurados.

¿El round-trip texto a IDs a texto es exacto?

Depende del tokenizer. No es una propiedad universal de los LLM, algunos reconstruyen el texto original carácter por carácter; otros aplican normalización y devuelven algo ligeramente distinto.


**Ida y vuelta (texto a ids a texto)**

In [ ]:
prueba = "Hola, ¿cómo estás?  "
print(f"Original : {repr(prueba)}")

vuelta = tok.decode(tok(prueba, add_special_tokens=False)["input_ids"])
print(f"  {REPO.split('/')[-1][:18]:<18}: {repr(vuelta)}  "
      f"{'IDÉNTICO' if prueba == vuelta else 'CAMBIÓ'}")

if tok_otro is not None:
    v2 = tok_otro.decode(tok_otro(prueba, add_special_tokens=False)["input_ids"])
    print(f"  {'TinyLlama':<18}: {repr(v2)}  "
          f"{'IDÉNTICO' if prueba == v2 else 'CAMBIÓ'}")

Original : 'Hola, ¿cómo estás?  '
  Qwen2.5-1.5B-Instr: 'Hola, ¿cómo estás?  '  IDÉNTICO
  TinyLlama         : 'Hola, ¿cómo estás?  '  IDÉNTICO


* Qwen usa un tokenizer BPE basado en bytes, diseñado para preservar la información del texto en el encode/decode.

* Otros tokenizadores pueden aplicar normalización durante la tokenización, por lo que el round-trip no necesariamente es idéntico al string original.


* Si el pipeline compara strings exactos o usa hashes para cachear, cambiar de tokenizer puede invalidar el cache sin producir ningún error. Por eso el tokenizer también debe considerarse parte del artefacto que se versiona. Ya que cambiar el tokenizer puede cambiar la representación del mismo texto y, por consecuencia, cambiar las claves del cache.

##**f) Etapa 4, Forward, distribución sobre todo el vocabulario**

El modelo no genera texto, genera para cada posición, un número por cada token del vocabulario (un "logit"). Eso es todo lo que hace. Elegir cuál de esos ~150.000 tokens sale es una decisión nuestra: el sampling.

In [ ]:
with torch.no_grad():
    salida = modelo(input_ids)

Última posición = el próximo token

In [ ]:
logits = salida.logits[0, -1, :].float()

In [ ]:
print(f"Shape de los logits: {tuple(salida.logits.shape)}  (batch, posición, vocabulario)")
print(f"Nos quedamos con: la última posición a vector de {logits.numel():,} números")

Shape de los logits: (1, 115, 151936)  (batch, posición, vocabulario)
Nos quedamos con: la última posición a vector de 151,936 números


El modelo procesa los 120 tokens y, en la última posición, produce 151.936 logits: uno por cada posible token que podría generar a continuación.

In [ ]:
probs = torch.softmax(logits, dim=-1)
top = torch.topk(probs, 10)

Los 10 candidatos más probables para el próximo token:

In [ ]:
print(f"  {'#':>2}  {'id':>7}  {'token':<20} {'prob':>8}   distribución")
print("  " + "-" * 66)
for r, (p, i) in enumerate(zip(top.values.tolist(), top.indices.tolist()), 1):
    barra = "█" * max(1, int(p * 40))
    print(f"  {r:>2}  {i:>7}  {repr(tok.decode([i])):<20} {p:>7.2%}   {barra}")

   #       id  token                    prob   distribución
  ------------------------------------------------------------------
   1     6582  'El'                  23.31%   █████████
   2    21999  'Si'                  11.36%   ████
   3    10580  'Cor'                  9.87%   ███
   4       50  'S'                    7.45%   ██
   5    17360  'Es'                   6.37%   ██
   6     8747  'La'                   3.80%   █
   7     2753  'No'                   3.63%   █
   8      693  'Re'                   3.36%   █
   9     1061  'Res'                  2.70%   █
  10     4262  'Lo'                   2.66%   █


In [ ]:
masa10 = top.values.sum().item()
print(f"\n  Esos 10 tokens concentran el {masa10:.1%} de la probabilidad.")
print(f"  Los otros {logits.numel()-10:,} se reparten el {1-masa10:.1%} restante.")


  Esos 10 tokens concentran el 74.5% de la probabilidad.
  Los otros 151,926 se reparten el 25.5% restante.


Todo lo que llamamos "parámetros de sampling" opera sobre este vector.

##**f) Etapa 5a, Temperatura**

La temperatura divide los logits antes del softmax:  softmax(logits / T)
* T < 1, agranda las diferencias: más determinista, más aburrido
* T = 1, la distribución tal cual la aprendió el modelo
* T > 1, aplana las diferencias: más variado, más riesgo de disparate
* T = 0, no está definido matemáticamente (división por cero).


Las librerías lo interpretan como "greedy": elegir siempre el máximo.

In [ ]:
def aplicar_temperatura(lg, T):
    """T<=0 se trata como greedy"""
    return lg if T is None or T <= 0 else lg / T

Temperatura sobre la misma distribución.

Este código toma los 6 tokens más probables y muestra cómo cambia su probabilidad cuando pasamos los logits por distintas temperaturas.

In [ ]:
TEMPS = [0.1, 0.7, 1.0, 1.5, 2.5]

print(f"  {'token':<18}", end="")
for T in TEMPS:
    print(f"{'T=' + str(T):>9}", end="")
print()
print("  " + "-" * (18 + 9 * len(TEMPS)))

ids_top = top.indices.tolist()[:6]
tablas  = {T: torch.softmax(aplicar_temperatura(logits, T), dim=-1) for T in TEMPS}

for i in ids_top:
    print(f"  {repr(tok.decode([i]))[:17]:<18}", end="")
    for T in TEMPS:
        print(f"{tablas[T][i].item():>8.1%} ", end="")
    print()

  token                 T=0.1    T=0.7    T=1.0    T=1.5    T=2.5
  ---------------------------------------------------------------
  'El'                 99.9%    39.3%    23.3%     9.9%     1.3% 
  'Si'                  0.1%    14.1%    11.4%     6.2%     1.0% 
  'Cor'                 0.0%    11.5%     9.9%     5.6%     0.9% 
  'S'                   0.0%     7.7%     7.4%     4.6%     0.8% 
  'Es'                  0.0%     6.2%     6.4%     4.2%     0.8% 
  'La'                  0.0%     3.0%     3.8%     3.0%     0.6% 


Con T=0.1 la decisión ya está tomada. Con T=2.5 casi cualquier token del vocabulario es posible. La temperatura no 'mejora' el modelo: redistribuye el riesgo entre repetitivo y delirante.

##**g) Etapa 5b, Top-K y Top-P (recortes del espacio de candidatos)**

Ambos hacen lo mismo conceptualmente: tirar a un lado la cola de tokens improbables antes de sortear. Pero con criterios distintos:

* top-k = 50, "quedate con los 50 mejores" (número fijo)
* top-p = 0.9, "quedate con los que sumen 90% de (número variable "probabilidad acumulada" según el contexto)

In [ ]:
def filtrar_top_k(lg, k):
    """Deja exactamente k candidatos vivos."""
    if not k or k <= 0 or k >= lg.numel():
        return lg
    idx = torch.topk(lg, k).indices
    out = torch.full_like(lg, -float("inf"))
    out[idx] = lg[idx]
    return out


def filtrar_top_p(lg, p):
    """Deja el conjunto mínimo cuya masa acumulada alcanza p."""
    if p is None or p >= 1.0:
        return lg
    orden = torch.argsort(lg, descending=True)
    pr    = torch.softmax(lg[orden], dim=-1)
    acum  = torch.cumsum(pr, dim=-1)
    # Descartar los que quedan después de superar p. El primero siempre sobrevive.
    descartar = (acum - pr) > p
    out = lg.clone()
    out[orden[descartar]] = -float("inf")
    return out


def cuantos_vivos(lg):
    return int(torch.isfinite(lg).sum().item())

Cuantos tokens sobreviven al filtro

In [ ]:
print(f"  vocabulario completo: {logits.numel():,} candidatos\n")
for k in (1, 5, 20, 50, 200):
    print(f"  top_k = {k:<4}: {cuantos_vivos(filtrar_top_k(logits, k)):>9,} candidatos")
print()
for p in (0.5, 0.8, 0.9, 0.95, 0.99):
    print(f"  top_p = {p:<4}: {cuantos_vivos(filtrar_top_p(logits, p)):>9,} candidatos")

  vocabulario completo: 151,936 candidatos

  top_k = 1   :         1 candidatos
  top_k = 5   :         5 candidatos
  top_k = 20  :        20 candidatos
  top_k = 50  :        50 candidatos
  top_k = 200 :       200 candidatos

  top_p = 0.5 :         4 candidatos
  top_p = 0.8 :        13 candidatos
  top_p = 0.9 :        24 candidatos
  top_p = 0.95:        45 candidatos
  top_p = 0.99:       168 candidatos


Orden de aplicación, acá hay una trampa real:

In [ ]:
lg_a = filtrar_top_p(filtrar_top_k(logits, 20), 0.9) # k y después p
lg_b = filtrar_top_k(filtrar_top_p(logits, 0.9), 20) # p y después k
lg_c = filtrar_top_p(aplicar_temperatura(logits, 1.5), 0.9) # T y después p
lg_d = aplicar_temperatura(filtrar_top_p(logits, 0.9), 1.5) # p y después T

In [ ]:
print(f"top_k(20) → top_p(0.9) : {cuantos_vivos(lg_a):>4} candidatos")
print(f"top_p(0.9) → top_k(20) : {cuantos_vivos(lg_b):>4} candidatos")
print(f"temperatura(1.5) → top_p(0.9) : {cuantos_vivos(lg_c):>4} candidatos")
print(f"top_p(0.9) → temperatura(1.5) : {cuantos_vivos(lg_d):>4} candidatos")

top_k(20) → top_p(0.9) :   13 candidatos
top_p(0.9) → top_k(20) :   20 candidatos
temperatura(1.5) → top_p(0.9) :  325 candidatos
top_p(0.9) → temperatura(1.5) :   24 candidatos


Los mismos parámetros, distinto orden, distinto resultado.


Hugging Face aplica temperatura antes de los filtros. Otros runtimes no. Copiar temperature=0.8, top_p=0.9 de un sistema a otro no garantiza el mismo comportamiento.

El runtime es campo del artefacto. No alcanza con registrar los parámetros: hay que registrar quién los interpreta.

##**h) Etapa 5c, Penalizaciones**

Tres fórmulas distintas para el mismo problema, que no son intercambiables:

| Parámetro | Origen | Fórmula | Naturaleza |
|---|---|---|---|
| `frequency_penalty` | OpenAI | `logit -= α · conteo` | proporcional al uso |
| `presence_penalty` | OpenAI | `logit -= β` si apareció | plano: una vez = veinte |
| `repetition_penalty` | HF / CTRL | `logit>0 → /r` ; `logit<0 → ×r` | multiplicativo, no lineal |

No existe traducción entre ellas. `frequency_penalty=0.5` no tiene equivalente en `repetition_penalty`. Y HF `generate()` no implementa frequency_penalty.

In [ ]:
def penalizar_frecuencia_presencia(lg, conteos, alfa_frec=0.0, beta_pres=0.0):
    """Estilo OpenAI: resta"""
    out = lg.clone()
    for tid, n in conteos.items():
        out[tid] -= alfa_frec * n + beta_pres * (1.0 if n > 0 else 0.0)
    return out


def penalizar_repeticion_hf(lg, conteos, r=1.0):
    """Estilo HuggingFace/CTRL: divide si el logit es positivo, multiplica si es negativo"""
    if r == 1.0:
        return lg
    out = lg.clone()
    for tid in conteos:
        out[tid] = out[tid] / r if out[tid] > 0 else out[tid] * r
    return out

Simulamos un historial donde un token ya salió 4 veces.

In [ ]:
tid_rep = ids_top[0]
conteos_demo = {tid_rep: 4}

print(f"  token bajo prueba: {repr(tok.decode([tid_rep]))}   "
      f"logit original: {logits[tid_rep]:.3f}\n")
print(f"  {'penalización':<38}{'logit':>10}{'prob':>10}")
print("  " + "-" * 58)

def _linea(etq, lg_p):
    p = torch.softmax(lg_p, dim=-1)[tid_rep].item()
    print(f"  {etq:<38}{lg_p[tid_rep].item():>10.3f}{p:>10.2%}")

_linea("sin penalización", logits)
_linea("frequency_penalty = 0.5 (×4 usos)",
       penalizar_frecuencia_presencia(logits, conteos_demo, alfa_frec=0.5))
_linea("frequency_penalty = 1.0 (×4 usos)",
       penalizar_frecuencia_presencia(logits, conteos_demo, alfa_frec=1.0))
_linea("presence_penalty  = 1.0 (plano)",
       penalizar_frecuencia_presencia(logits, conteos_demo, beta_pres=1.0))
_linea("repetition_penalty = 1.1 (HF)",
       penalizar_repeticion_hf(logits, conteos_demo, 1.1))
_linea("repetition_penalty = 1.5 (HF)",
       penalizar_repeticion_hf(logits, conteos_demo, 1.5))

  token bajo prueba: 'El'   logit original: 20.719

  penalización                               logit      prob
  ----------------------------------------------------------
  sin penalización                          20.719    23.31%
  frequency_penalty = 0.5 (×4 usos)         18.719     3.95%
  frequency_penalty = 1.0 (×4 usos)         16.719     0.55%
  presence_penalty  = 1.0 (plano)           19.719    10.06%
  repetition_penalty = 1.1 (HF)             18.835     4.42%
  repetition_penalty = 1.5 (HF)             13.812     0.03%


Cada una modifica los logits de forma distinta y, después del softmax, produce probabilidades muy distintas. Migrar de proveedor obliga a re-calibrar, no a traducir.

##**i) Etapa 6, Bucle Completo**

Esto es lo que hace `generate()` por dentro:
forward → penalizar → temperatura → filtrar → sortear → agregar el token → repetir.

Dos detalles que corrigen errores frecuentes:

1. **Los conteos se inicializan con el prompt.** El `RepetitionPenaltyLogitsProcessor`
   de HF penaliza sobre la secuencia completa, prompt incluido. Contar solo los tokens
   generados **no** reproduce a HF.
2. **El corte usa el conjunto `EOS_IDS`**, no un único id.

In [ ]:
@torch.no_grad()
def generar_a_mano(ids_entrada, max_new_tokens=60, temperatura=0.7,
                   top_k=None, top_p=None, frequency_penalty=0.0,
                   presence_penalty=0.0, repetition_penalty=1.0,
                   seed=None, verboso=0):
    """Versión didáctica: SIN KV cache (ver la sección siguiente)."""
    if seed is not None:
        torch.manual_seed(seed)

    seq = ids_entrada.clone()

    # HF penaliza también los tokens del prompt.
    conteos = {}
    for t in ids_entrada[0].tolist():
        conteos[t] = conteos.get(t, 0) + 1

    generados = []

    for paso in range(max_new_tokens):
        # Sin cache: se re-procesa toda la secuencia en cada paso.
        lg = modelo(seq).logits[0, -1, :].float()

        # Este es el orden de HF: processors, después warpers
        lg = penalizar_repeticion_hf(lg, conteos, repetition_penalty)
        lg = penalizar_frecuencia_presencia(lg, conteos, frequency_penalty, presence_penalty)

        if temperatura is None or temperatura <= 0:
            siguiente = int(torch.argmax(lg)) # greedy
        else:
            lg = aplicar_temperatura(lg, temperatura)
            lg = filtrar_top_k(lg, top_k)
            lg = filtrar_top_p(lg, top_p)
            pr = torch.softmax(lg, dim=-1)
            siguiente = int(torch.multinomial(pr, num_samples=1))

        if verboso and paso < verboso:
            p = torch.softmax(lg, dim=-1)[siguiente].item()
            print(f"    paso {paso:>2}: id={siguiente:>6}  "
                  f"{repr(tok.decode([siguiente])):<14} p={p:>6.1%}  "
                  f"candidatos vivos={cuantos_vivos(lg):,}")

        if siguiente in EOS_IDS: # conjunto, no un id suelto
            break

        conteos[siguiente] = conteos.get(siguiente, 0) + 1
        generados.append(siguiente)
        seq = torch.cat([seq, torch.tensor([[siguiente]], device=seq.device)], dim=1)

    # ETAPA 7: detokenización
    return tok.decode(generados, skip_special_tokens=True), generados

El bucle (primeros 8 tokens visibles)

In [ ]:
txt, gen_ids = generar_a_mano(input_ids, max_new_tokens=70, temperatura=0.7,
                              top_p=0.9, top_k=50, seed=42, verboso=8)

    paso  0: id=  6582  'El'           p= 43.4%  candidatos vivos=10
    paso  1: id= 26629  ' cliente'     p= 17.8%  candidatos vivos=3
    paso  2: id= 34374  ' debe'        p= 52.7%  candidatos vivos=5
    paso  3: id=  3645  ' contact'     p= 66.3%  candidatos vivos=9
    paso  4: id=   277  'ar'           p=100.0%  candidatos vivos=1
    paso  5: id=  1187  ' la'          p=  7.1%  candidatos vivos=4
    paso  6: id=  8988  ' ti'          p=100.0%  candidatos vivos=1
    paso  7: id=  9696  'enda'         p=100.0%  candidatos vivos=1


In [ ]:
print(f"Tokens generados: {len(gen_ids)}")
print("Texto final:")
print(textwrap.indent(textwrap.fill(txt, 72), "    "))

Tokens generados: 38
Texto final:
    El cliente debe contactar la tienda para informar el problema y
    solicitar un servicio técnico. Si no se soluciona, puede considerar una
    devolución o una compensación.


##**j) KV cache**

`generar_a_mano` llama `modelo(seq)` con la secuencia **completa** en cada paso.
Eso recalcula las proyecciones K y V de todos los tokens anteriores una y otra vez:
el decode termina siendo **cuadrático**.

En cada paso el modelo necesita atender a todos los tokens previos, pero sus K y V
**ya fueron calculados**. El KV cache los guarda y los reutiliza: solo se procesa
el token nuevo.

Esto no es una optimización menor. Es la razón por la que:

- la memoria de inferencia crece con `contexto × batch`
- el límite de concurrencia de un servidor casi nunca son los pesos, es el KV cache
- existen vLLM y PagedAttention

Midámoslo.

In [ ]:
@torch.no_grad()
def generar_con_cache(ids_entrada, max_new_tokens=60, temperatura=0.7,
                      top_k=None, top_p=None, seed=None):
    """Misma lógica de sampling, pero reutilizando el KV cache."""
    if seed is not None:
        torch.manual_seed(seed)

    # Prefill: una sola pasada por todo el prompt, devuelve el cache inicial.
    out  = modelo(ids_entrada, use_cache=True)
    past = out.past_key_values
    lg   = out.logits[0, -1, :].float()

    generados = []
    for _ in range(max_new_tokens):
        if temperatura is None or temperatura <= 0:
            siguiente = int(torch.argmax(lg))
        else:
            l = filtrar_top_p(filtrar_top_k(aplicar_temperatura(lg, temperatura), top_k), top_p)
            siguiente = int(torch.multinomial(torch.softmax(l, dim=-1), num_samples=1))

        if siguiente in EOS_IDS:
            break
        generados.append(siguiente)

        # Decode: solo el token nuevo, lo anterior vive en el cache.
        out  = modelo(torch.tensor([[siguiente]], device=DEVICE),
                      past_key_values=past, use_cache=True)
        past = out.past_key_values
        lg   = out.logits[0, -1, :].float()

    return tok.decode(generados, skip_special_tokens=True), generados

In [ ]:
N = 60
print(f"Prompt: {input_ids.shape[1]} tokens · generando {N} tokens · greedy\n")

resultados = {}
for fn, etq in ((generar_a_mano, "Sin KV cache"), (generar_con_cache, "Con KV cache")):
    t0 = time.perf_counter()
    t, g = fn(input_ids, max_new_tokens=N, temperatura=0.0)
    dt = time.perf_counter() - t0
    resultados[etq] = (dt, g)
    print(f"  {etq:<14}: {dt:6.2f} s ({len(g)} tokens → {dt/max(len(g),1)*1000:6.1f} ms/token)")

speedup = resultados["Sin KV cache"][0] / resultados["Con KV cache"][0]
print(f"\n  speedup: {speedup:.1f}×")
print(f"  ¿mismo resultado? {resultados['Sin KV cache'][1] == resultados['Con KV cache'][1]}")

Prompt: 115 tokens · generando 60 tokens · greedy

  Sin KV cache  :   1.69 s (33 tokens →   51.2 ms/token)
  Con KV cache  :   1.78 s (33 tokens →   54.1 ms/token)

  speedup: 0.9×
  ¿mismo resultado? True


En esta prueba el KV Cache no produjo un speedup porque el contexto y la cantidad de tokens generados son relativamente pequeños, por lo que el costo de gestionar la caché puede compensar el ahorro de cómputo. Sin embargo, ambas configuraciones producen exactamente el mismo resultado. La ventaja del KV Cache se vuelve más evidente a medida que aumenta el contexto y la cantidad de tokens generados.

---

Mismo texto, mismo modelo, misma seed, solo cambia dónde vive el estado y ese estado es memoria de GPU que no son los pesos.

* Con un prompt de 120 tokens la diferencia ya se nota.
* Con 4.000 tokens de contexto RAG, la versión sin cache es inutilizable.
* Este es el argumento por el cual el KV cache, y no los pesos, define cuántos usuarios concurrentes soporta una GPU.

**El mismo prompt con distintos parámetros**

Mismo modelo, misma revisión, mismo prompt, lo único que cambia es el sampling.

In [ ]:
CONFIGS = [
    ("greedy (T=0)",             dict(temperatura=0.0)),
    ("T=0.7 top_p=0.9 seed=42",  dict(temperatura=0.7, top_p=0.9, seed=42)),
    ("T=0.7 top_p=0.9 seed=7",   dict(temperatura=0.7, top_p=0.9, seed=7)),
    ("T=1.5 top_p=0.95 seed=42", dict(temperatura=1.5, top_p=0.95, seed=42)),
]

print(f"El mismo prompt, mismo modelo y mismo SHA, {len(CONFIGS)} configuraciones")

for etq, kw in CONFIGS:
    t, g = generar_con_cache(input_ids, max_new_tokens=60, **kw)
    unicos = len(set(g))
    print(f"\n - {etq}   ({len(g)} tokens, {unicos} distintos "
          f"= {unicos/max(len(g),1):.0%} de variedad léxica)")
    print(textwrap.indent(textwrap.fill(t.strip(), 70), "     "))

El mismo prompt, mismo modelo y mismo SHA, 4 configuraciones

 - greedy (T=0)   (33 tokens, 28 distintos = 85% de variedad léxica)
     El reclamo corresponde a gestionar. El cliente debe contactar al
     servicio al cliente de la tienda para informar el problema y solicitar
     una solución.

 - T=0.7 top_p=0.9 seed=42   (38 tokens, 34 distintos = 89% de variedad léxica)
     El cliente debe contactar la tienda para informar el problema y
     solicitar un servicio técnico. Si no se soluciona, puede considerar
     una devolución o una compensación.

 - T=0.7 top_p=0.9 seed=7   (52 tokens, 39 distintos = 75% de variedad léxica)
     Si corresponde gestionar el reclamo, el cliente debe verificar si el
     producto fue entregado en perfectas condiciones y si el problema se
     produjo durante el uso. Puede contactar al servicio al cliente para
     una revisión y una solución al problema.

 - T=1.5 top_p=0.95 seed=42   (60 tokens, 59 distintos = 98% de variedad léxica)
     Bonita d

Las respuestas que ve el cliente son distintas, por eso los parámetros de sampling son campo del artefacto.

##**k) El artefacto completo**

Las  etapas recorridas son  campos que hay que poder reconstruir. Los juntamos en un solo objeto: entorno.json.

Este archivo es exactamente lo que la tabla "Registro en LLMOps" y es el input del ejercicio de MLflow.

Con esto respondemos las cuatro preguntas del registry:

* ¿Qué versiones de este sistema existen?
* ¿Cuál vale ahora, y desde cuándo?
* ¿Con qué evidencia se promovió?
* ¿A cuál vuelvo si falla, y cuánto tarda?

In [ ]:
SAMPLING = {"temperature": 0.7, "top_p": 0.9, "top_k": 50,
            "max_new_tokens": 120, "seed": 42}

artefacto = {
    # 1 y 2, modelo y tokenizer, con revisión inmutable
    "modelo":    {"repo": REPO, "revision": SHA_EN_DISCO},
    "tokenizer": {"repo": REPO, "revision": SHA_EN_DISCO},

    # 3, chat template: no es nuestro, así que lo copiamos y lo hasheamos
    "chat_template_sha256": hashlib.sha256(
        (tok.chat_template or "").encode()
    ).hexdigest()[:12],

    # 4, prompt template: es nuestro, vive en git, se hashea
    "prompt_template_sha256": hash_prompt,

    # 5, sampling explícito + los defaults que venían del proveedor
    "sampling": SAMPLING,
    "generation_config_defaults": modelo.generation_config.to_diff_dict(),

    # 6, runtime y hardware
    "runtime": {
        "python":       platform.python_version(),
        "torch":        torch.__version__,
        "transformers": transformers.__version__,
        "device":       DEVICE,
        "dtype":        str(DTYPE),
        "gpu":          torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
        "cuda":         torch.version.cuda,
    },

    # 7, recuperación: vacío en este ejercicio, presente en el de RAG
    "recuperacion": None,

}

with open("entorno.json", "w", encoding="utf-8") as f:
    json.dump(artefacto, f, indent=2, ensure_ascii=False)

print(json.dumps(artefacto, indent=2, ensure_ascii=False))

{
  "modelo": {
    "repo": "Qwen/Qwen2.5-1.5B-Instruct",
    "revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
  },
  "tokenizer": {
    "repo": "Qwen/Qwen2.5-1.5B-Instruct",
    "revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
  },
  "chat_template_sha256": "cd8e9439f057",
  "prompt_template_sha256": "f711814fec4a",
  "sampling": {
    "temperature": 0.7,
    "top_p": 0.9,
    "top_k": 50,
    "max_new_tokens": 120,
    "seed": 42
  },
  "generation_config_defaults": {
    "do_sample": true,
    "temperature": 0.7,
    "top_k": 20,
    "top_p": 0.8,
    "repetition_penalty": 1.1,
    "pad_token_id": 151643,
    "bos_token_id": 151643,
    "eos_token_id": [
      151645,
      151643
    ],
    "transformers_version": "5.16.1"
  },
  "runtime": {
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "transformers": "5.16.1",
    "device": "cuda",
    "dtype": "torch.float16",
    "gpu": "Tesla T4",
    "cuda": "12.8"
  },
  "recuperacion": null,
  "evaluacion": {
    "m

## Cierre

Recorrimos las ocho etapas que esconde `modelo.generate(prompt)` y, en cada una,
encontramos algo que puede cambiar sin que toquemos una línea de nuestro código:

| Etapa | Qué puede cambiar solo | Cómo lo fijamos |
|---|---|---|
| 1. Prompt template | Nosotros | hash + git |
| 2. Chat template | Un commit ajeno | `revision` + copia hasheada |
| 3. Tokenización | Cambio de modelo | va con la revisión |
| 4. Prefill | `dtype`, hardware | `entorno.json` |
| 5. Sampling | `generation_config.json` del proveedor | parámetros explícitos |
| 6. Decode | Versión del runtime | versión pinneada |
| 7. Detokenización | Normalización del tokenizer | va con la revisión |

El notebook produjo un `json_valid_rate` hoy. ¿Podemos volver a esa configuración exacta la semana que viene, en otra máquina, sin adivinar?

Esto es **model registry**.